In [16]:

import pandas as pd
import numpy as np
import re
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


In [17]:
def clean_name(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z\s]", "", x)
    return re.sub(r"\s+", " ", x).strip()
def extract_region(area_str, keyword):
    after = area_str.split(keyword, 1)[1].strip()
    return clean_name(re.sub(r"\s+\d+$", "", after))
def parse_census_appendix(file_path, filter_col, filter_val, value_col, out_col):
    df = pd.read_excel(file_path, skiprows=5, header=None)
    df.columns = range(len(df.columns))
    rows, seen, current_state, current_district = [], set(), None, None
    for _, row in df.iterrows():
        area  = str(row[3]) if pd.notna(row[3]) else ""
        grp   = str(row[filter_col]) if pd.notna(row[filter_col]) else ""
        cat   = str(row[6]) if pd.notna(row[6]) else ""
        value = row[value_col]
        if "STATE -" in area:
            current_state    = extract_region(area, "STATE -")
            current_district = None
        if "District -" in area:
            current_district = extract_region(area, "District -")
        if grp == filter_val and cat == "Total":
            key = (current_state, current_district)
            if current_state and current_district and key not in seen:
                rows.append([current_state, current_district, value])
                seen.add(key)
    return pd.DataFrame(rows, columns=["state", "district", out_col])

In [18]:
df = pd.read_excel(
    r"C:\Users\HP\Downloads\A-1_NO_OF_VILLAGES_TOWNS_HOUSEHOLDS_POPULATION_AND_AREA.xlsx",
    skiprows=5
)
df.columns = [
    "state_code","district_code","subdistrict_code","level","name","area_type",
    "villages_total","villages_inhabited","villages_uninhabited","towns",
    "households","population_total","male","female","area"
]
df["state_code"]       = df["state_code"].astype(str).str.zfill(2)
df["level"]            = df["level"].astype(str).str.strip().str.upper()
df["area_type"]        = df["area_type"].astype(str).str.strip().str.title()
df["name"]             = df["name"].astype(str).str.strip()
df["population_total"] = df["population_total"].astype(str).str.replace(",", "").astype(float)
state_map = (
    df[df["level"] == "STATE"][["state_code","name"]]
    .drop_duplicates()
    .assign(name=lambda x: x["name"].apply(clean_name))
    .set_index("state_code")["name"]
    .to_dict()
)
df = df[df["level"] == "DISTRICT"].copy()
df["state"]    = df["state_code"].map(state_map)
df["district"] = df["name"].apply(clean_name)
census_df = (
    df[["state","district","area_type","population_total"]]
    .pivot_table(index=["state","district"], columns="area_type",
                 values="population_total", aggfunc="sum")
    .reset_index()
    .rename(columns={"Total":"total_population","Rural":"rural_population","Urban":"urban_population"})
)
census_df.columns.name = None
print(census_df.head())

                     state              district  rural_population  \
0  andaman nicobar islands              nicobars           20727.0   
1  andaman nicobar islands  north middle andaman           53457.0   
2  andaman nicobar islands         south andaman           52103.0   
3           andhra pradesh              adilabad          985303.0   
4           andhra pradesh             anantapur         1489157.0   

   total_population  urban_population  
0           20727.0               0.0  
1           54861.0            1404.0  
2          127283.0           75180.0  
3         1369597.0          384294.0  
4         2064495.0          575338.0  


In [19]:
df_final_st = parse_census_appendix(
    r"C:\Users\HP\Downloads\ST-27-PCA-A11-APPENDIX.xlsx",
    filter_col=5, filter_val="All Schedule Tribes",
    value_col=7, out_col="st_total"
)
print(df_final_st.to_string(index=False))

      state        district  st_total
maharashtra       nandurbar 1141933.0
maharashtra           dhule  647315.0
maharashtra         jalgaon  604367.0
maharashtra         buldana  124837.0
maharashtra           akola  100280.0
maharashtra          washim   80471.0
maharashtra        amravati  404128.0
maharashtra          wardha  149507.0
maharashtra          nagpur  437571.0
maharashtra        bhandara   88886.0
maharashtra         gondiya  214253.0
maharashtra      gadchiroli  415306.0
maharashtra      chandrapur  389441.0
maharashtra        yavatmal  514057.0
maharashtra          nanded  281695.0
maharashtra         hingoli  111954.0
maharashtra        parbhani   40514.0
maharashtra           jalna   42263.0
maharashtra      aurangabad  143366.0
maharashtra          nashik 1564369.0
maharashtra           thane 1542451.0
maharashtra mumbai suburban  104560.0
maharashtra          mumbai   25093.0
maharashtra         raigarh  305125.0
maharashtra            pune  348876.0
maharashtra 

In [20]:
df_final_sc = parse_census_appendix(
    r"C:\Users\HP\Downloads\SC-27-PCA-A10-APPENDIX.xlsx",
    filter_col=5, filter_val="All Schedule Castes",
    value_col=7, out_col="sc_total"
)
print(df_final_sc.to_string(index=False))

      state        district  sc_total
maharashtra       nandurbar   47985.0
maharashtra           dhule  127571.0
maharashtra         jalgaon  389273.0
maharashtra         buldana  470895.0
maharashtra           akola  364059.0
maharashtra          washim  229462.0
maharashtra        amravati  506374.0
maharashtra          wardha  188830.0
maharashtra          nagpur  867713.0
maharashtra        bhandara  200372.0
maharashtra         gondiya  175961.0
maharashtra      gadchiroli  120745.0
maharashtra      chandrapur  348365.0
maharashtra        yavatmal  328518.0
maharashtra          nanded  640483.0
maharashtra         hingoli  182565.0
maharashtra        parbhani  247308.0
maharashtra           jalna  272266.0
maharashtra      aurangabad  539368.0
maharashtra          nashik  554687.0
maharashtra           thane  730089.0
maharashtra mumbai suburban  583302.0
maharashtra          mumbai  219934.0
maharashtra         raigarh  134952.0
maharashtra            pune 1180703.0
maharashtra 

In [21]:
df = pd.read_excel(r"C:\Users\HP\Downloads\DDW-2700C-08.xlsx", skiprows=5, header=None)
df.columns = range(len(df.columns))
rows, seen, current_state, current_district = [], set(), None, None
for _, row in df.iterrows():
    area = str(row[3]) if pd.notna(row[3]) else ""
    tru  = str(row[4]) if pd.notna(row[4]) else ""
    age  = str(row[5]) if pd.notna(row[5]) else ""
    if "State -" in area:
        current_state    = extract_region(area, "State -")
        current_district = None
    if "District -" in area:
        current_district = extract_region(area, "District -")
    if tru == "Total" and age == "All ages":
        total, literate = row[6], row[12]
        key = (current_state, current_district)
        if pd.notna(total) and pd.notna(literate) and current_state and current_district and key not in seen:
            rows.append([current_state, current_district, float(literate) / float(total)])
            seen.add(key)
df_literacy = pd.DataFrame(rows, columns=["state","district","literacy_rate"])
print(df_literacy.head())

         state   district  literacy_rate
0  maharashtra  nandurbar       0.549968
1  maharashtra      dhule       0.630913
2  maharashtra    jalgaon       0.683673
3  maharashtra    buldana       0.726870
4  maharashtra      akola       0.778034


In [22]:
nfhs = pd.read_excel(r"C:\Users\HP\Downloads\NFHS_5_India_Districts_Factsheet_Data.xls", engine="xlrd")
nfhs.columns = nfhs.columns.str.lower().str.strip()
nfhs = nfhs.rename(columns={
    "state/ut": "state",
    "district names": "district",
    "women (age 15-49)  with 10 or more years of schooling (%)": "female_edu_pct",
    "households using clean fuel for cooking3 (%)": "clean_fuel_pct"
})
nfhs["state"]    = nfhs["state"].apply(clean_name).replace({"maharastra": "maharashtra"})
nfhs["district"] = nfhs["district"].apply(clean_name)
nfhs["female_edu_pct"] = pd.to_numeric(nfhs["female_edu_pct"], errors="coerce")
nfhs["clean_fuel_pct"] = pd.to_numeric(nfhs["clean_fuel_pct"], errors="coerce")
nfhs["deprivation_score"] = (
    0.5 * (1 - nfhs["female_edu_pct"] / 100) +
    0.5 * (1 - nfhs["clean_fuel_pct"] / 100)
)
nfhs_final = nfhs[["state","district","deprivation_score"]]

In [23]:
mgnrega_raw = pd.read_csv(
    r"C:\Users\HP\Downloads\ee03643a-ee4c-48c2-ac30-9f2ff26ab722_c5a484e50805c1f4d221632e2f3625f5.csv"
)
mgnrega_raw.columns = mgnrega_raw.columns.str.strip().str.lower().str.replace(" ", "_")
state_col    = [c for c in mgnrega_raw.columns if "state"    in c][0]
district_col = [c for c in mgnrega_raw.columns if "district" in c][0]
house_col    = [c for c in mgnrega_raw.columns if "house"    in c][0]
person_col   = [c for c in mgnrega_raw.columns if "person"   in c][0]
def clean_mgnrega(x):
    x = str(x).lower().strip()
    x = re.sub(r"\(.*?\)", "", x)
    x = re.sub(r"[^a-z ]", "", x)
    return x.replace("district", "").strip()
mgnrega = mgnrega_raw.copy()
mgnrega["state"]    = mgnrega[state_col].apply(clean_mgnrega)
mgnrega["district"] = mgnrega[district_col].apply(clean_mgnrega).replace({
    "ahmednagar": "ahmadnagar", "beed": "bid", "raigad": "raigarh",
    "mumbai city": "mumbai", "mumbai suburban district": "mumbai suburban"
})
mgnrega["total_households_worked"] = pd.to_numeric(mgnrega[house_col],  errors="coerce")
mgnrega["persondays"]              = pd.to_numeric(mgnrega[person_col], errors="coerce")
mgnrega["mgnrega_dependency"]      = mgnrega["persondays"] / mgnrega["total_households_worked"]
mgnrega = mgnrega[["state","district","mgnrega_dependency"]]

In [24]:
STATE = "maharashtra"
def filter_dedup(df_):
    return df_[df_["state"] == STATE].drop_duplicates(["state","district"])
census_df   = filter_dedup(census_df)
df_final_st = filter_dedup(df_final_st)
df_final_sc = filter_dedup(df_final_sc)
df_literacy = filter_dedup(df_literacy)
nfhs_final  = filter_dedup(nfhs_final)
final = census_df.copy()
for df_ in [df_final_st, df_final_sc, df_literacy, nfhs_final, mgnrega]:
    final = final.merge(df_, on=["state","district"], how="left")
final["st_pct"] = final["st_total"] / final["total_population"]
final["sc_pct"] = final["sc_total"] / final["total_population"]
final["mgnrega_dependency"] = final["mgnrega_dependency"].fillna(0)
final = final[[
    "state","district","total_population",
    "st_pct","sc_pct","literacy_rate","deprivation_score","mgnrega_dependency"
]]
print(final.shape)
print(final.head())

(35, 8)
         state    district  total_population    st_pct    sc_pct  \
0  maharashtra  ahmadnagar         2342825.0  0.161442  0.244874   
1  maharashtra       akola          932334.0  0.107558  0.390481   
2  maharashtra    amravati         1480768.0  0.272918  0.341967   
3  maharashtra  aurangabad         1924469.0  0.074496  0.280268   
4  maharashtra    bhandara          605520.0  0.146793  0.330909   

   literacy_rate  deprivation_score  mgnrega_dependency  
0       0.693766            0.38415                 0.0  
1       0.778034            0.35870                 0.0  
2       0.779615            0.36625                 0.0  
3       0.676478            0.36600                 0.0  
4       0.749675            0.37335                 0.0  


In [25]:
final["vulnerability_score"] = (
    0.3 * final["st_pct"] +
    0.2 * final["sc_pct"] +
    0.2 * (1 - final["literacy_rate"]) +
    0.3 * final["deprivation_score"]
)
vs = final["vulnerability_score"]
final["vulnerability_score"] = (vs - vs.min()) / (vs.max() - vs.min())
threshold = final["vulnerability_score"].quantile(0.75)
final["high_risk"] = (final["vulnerability_score"] >= threshold).astype(int)
print(final.head())

         state    district  total_population    st_pct    sc_pct  \
0  maharashtra  ahmadnagar         2342825.0  0.161442  0.244874   
1  maharashtra       akola          932334.0  0.107558  0.390481   
2  maharashtra    amravati         1480768.0  0.272918  0.341967   
3  maharashtra  aurangabad         1924469.0  0.074496  0.280268   
4  maharashtra    bhandara          605520.0  0.146793  0.330909   

   literacy_rate  deprivation_score  mgnrega_dependency  vulnerability_score  \
0       0.693766            0.38415                 0.0             0.269479   
1       0.778034            0.35870                 0.0             0.250044   
2       0.779615            0.36625                 0.0             0.320577   
3       0.676478            0.36600                 0.0             0.234102   
4       0.749675            0.37335                 0.0             0.266767   

   high_risk  
0          0  
1          0  
2          0  
3          0  
4          0  


In [26]:
features = ["st_pct", "sc_pct", "literacy_rate", "deprivation_score"]
X, y = final[features], final["high_risk"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = XGBClassifier()
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))
importance = pd.DataFrame({"feature": X.columns, "importance": model.feature_importances_})
print(importance.sort_values("importance", ascending=False))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         2

    accuracy                           1.00         7
   macro avg       1.00      1.00      1.00         7
weighted avg       1.00      1.00      1.00         7

             feature  importance
3  deprivation_score    0.415676
2      literacy_rate    0.352200
0             st_pct    0.168050
1             sc_pct    0.064074


In [27]:
def simulate_policy_cut(df, cut_pct):
    impact = cut_pct / 100
    sim = df.copy()
    sim["simulated_deprivation"] = sim["deprivation_score"] * (1 + impact)
    sim["simulated_vulnerability"] = (
        0.25 * sim["st_pct"] +
        0.15 * sim["sc_pct"] +
        0.20 * (1 - sim["literacy_rate"]) +
        0.20 * sim["simulated_deprivation"] +
        0.20 * sim["mgnrega_dependency"] * (1 + impact)
    )
    sv = sim["simulated_vulnerability"]
    sim["simulated_vulnerability"] = (sv - sv.min()) / (sv.max() - sv.min())
    return sim
sim = simulate_policy_cut(final, 20)
print(sim.sort_values("simulated_vulnerability", ascending=False).head())

          state    district  total_population    st_pct    sc_pct  \
20  maharashtra   nandurbar          833170.0  1.370588  0.057593   
9   maharashtra  gadchiroli          541328.0  0.767198  0.223053   
8   maharashtra       dhule         1054031.0  0.614133  0.121032   
21  maharashtra      nashik         3157186.0  0.495495  0.175690   
11  maharashtra     hingoli          606294.0  0.184653  0.301116   

    literacy_rate  deprivation_score  mgnrega_dependency  vulnerability_score  \
20       0.549968            0.64895                 0.0             1.000000   
9        0.660208            0.48610                 0.0             0.631227   
8        0.630913            0.47095                 0.0             0.521671   
21       0.711517            0.42475                 0.0             0.429590   
11       0.671632            0.56855                 0.0             0.400856   

    high_risk  simulated_deprivation  simulated_vulnerability  
20          1                0.778

In [28]:
final.to_csv(r"C:\Users\HP\OneDrive\Documents\govt\final_dataset.csv", index=False)